# Logistic Regression Model for Telecom Customer Churn
This notebook trains and evaluates the Logistic Regression model on our cleaned dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

## 1. Load Data

In [ ]:
train_df = pd.read_csv('../../data/processed/train.csv')
test_df = pd.read_csv('../../data/processed/test.csv')

train_df['Churn'] = train_df['Churn'].map({'Yes': 1, 'No': 0})
test_df['Churn'] = test_df['Churn'].map({'Yes': 1, 'No': 0})

X_train_raw = train_df.drop(columns=['Churn'])
y_train = train_df['Churn']
X_test_raw = test_df.drop(columns=['Churn'])
y_test = test_df['Churn']

## 2. One-Hot Encoding

In [ ]:
cat_cols = X_train_raw.select_dtypes(exclude='number').columns.tolist()
X_train = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test_raw, columns=cat_cols, drop_first=True)

# Align test columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print("Train features:", X_train.shape[1])

## 3. Fit Model

In [ ]:
from sklearn.linear_model import LogisticRegression
# Logistic regression requires scaled input
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
preds = model.predict(X_test_s)

## 4. Evaluation

In [ ]:
print(classification_report(y_test, preds))
if hasattr(model, 'predict_proba'):
    probs = model.predict_proba(X_test if 's' not in locals() else X_test_s)[:, 1]
    print('ROC AUC Score:', roc_auc_score(y_test, probs))
else:
    print('ROC AUC Score:', roc_auc_score(y_test, preds))